# Module 02-10 — Model Comparison & Algorithm Selection

**Learning objective:** Train every algorithm from Module 02 on the same dataset under identical conditions, then build a practical decision framework for choosing the right algorithm based on data size, dimensionality, interpretability, and latency requirements.

**Prerequisites:** 02-01 through 02-09 (all Module 02 algorithms)

> **Note:** This is a **comparison notebook** (Category F). We do *not* re-teach individual algorithms — we reference prior notebooks and focus entirely on head-to-head evaluation and the algorithm selection decision framework.

---

## Part 0 — Setup & Prerequisites

All algorithms are evaluated on the **scikit-learn Digits dataset** (1 797 samples, 64 features, 10 classes) with:
- Identical 80/20 train/test split (`SEED = 1103`)
- Identical preprocessing (`StandardScaler`)
- Same evaluation metrics (accuracy, macro-F1, training time, inference time)

The algorithms covered span the full Module 02 sequence:

| # | Algorithm | Notebook |
|---|-----------|----------|
| 1 | Majority Class Baseline | — |
| 2 | Ridge Classifier (linear) | 02-01 |
| 3 | Logistic Regression | 02-02 |
| 4 | Decision Tree | 02-03 |
| 5 | Random Forest | 02-03 |
| 6 | AdaBoost | 02-04 |
| 7 | Gradient Boosting | 02-04 |
| 8 | SVM (Linear kernel) | 02-05 |
| 9 | SVM (RBF kernel) | 02-05 |
| 10 | k-NN (k = 5) | 02-07 |
| 11 | Gaussian Naive Bayes | 02-08 |
| 12 | Soft Voting Ensemble | 02-09 |
| 13 | Stacking Ensemble | 02-09 |

In [ ]:
import sys
import time
import copy
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

from sklearn.datasets import load_digits
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

import torch
warnings.filterwarnings("ignore")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Configuration
TEST_SIZE   = 0.20   # 80/20 classical ML split — see Module 1-07
N_CV_FOLDS  = 5      # cross-validation folds for robustness check
N_REPEATS   = 3      # repeats for timing stability

In [ ]:
# Load Digits dataset and build train/test splits
digits = load_digits()
X_all, y_all = digits.data, digits.target

print(f"Dataset : sklearn Digits")
print(f"Samples : {X_all.shape[0]}  |  Features: {X_all.shape[1]}  |  Classes: {len(np.unique(y_all))}")
print(f"Class distribution: {dict(Counter(y_all))}")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_all,
)

# Standardise — important for SVM, k-NN, Logistic Regression
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f"\nTrain: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

# Quick EDA: mean digit images per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in enumerate(axes.flat):
    mask = y_all == digit
    mean_img = X_all[mask].mean(axis=0).reshape(8, 8)
    ax.imshow(mean_img, cmap="gray_r")
    ax.set_title(f"Digit {digit}  (n={mask.sum()})")
    ax.axis("off")
plt.suptitle("Mean image per class — Digits Dataset", fontsize=12)
plt.tight_layout()
plt.show()

---

## Part 1 — Defining All Algorithms

Each algorithm is instantiated with hyperparameters chosen to represent its typical "out-of-the-box" performance. We do not tune per-algorithm — the goal is a fair baseline comparison.

References: algorithms are fully derived and implemented in their respective notebooks (see table in Part 0). We reference rather than re-teach.

In [ ]:
def build_algorithm_registry() -> list:
    """Build the ordered list of (name, classifier, meta) tuples for comparison.

    Each entry contains a display name, a fresh sklearn estimator, and a
    metadata dict describing key properties for the decision framework.

    Returns:
        List of (name, estimator, metadata_dict) tuples.
    """
    registry = [
        (
            "Majority Baseline",
            DummyClassifier(strategy="most_frequent", random_state=SEED),
            {"notebook": "—",     "family": "Baseline",   "interpretable": True,
             "needs_proba": False, "n_params": 0,           "notes": "Random chance upper bound"},
        ),
        (
            "Ridge Classifier",
            RidgeClassifier(alpha=1.0),
            {"notebook": "02-01", "family": "Linear",      "interpretable": True,
             "needs_proba": False, "n_params": None,         "notes": "Linear — no prob output"},
        ),
        (
            "Logistic Regression",
            LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
            {"notebook": "02-02", "family": "Linear",      "interpretable": True,
             "needs_proba": True,  "n_params": None,         "notes": "Well-calibrated probs"},
        ),
        (
            "Decision Tree",
            DecisionTreeClassifier(max_depth=10, random_state=SEED),
            {"notebook": "02-03", "family": "Tree",        "interpretable": True,
             "needs_proba": True,  "n_params": None,         "notes": "Fully interpretable"},
        ),
        (
            "Random Forest",
            RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
            {"notebook": "02-03", "family": "Ensemble",    "interpretable": False,
             "needs_proba": True,  "n_params": None,         "notes": "Low variance, parallelisable"},
        ),
        (
            "AdaBoost",
            AdaBoostClassifier(n_estimators=100, random_state=SEED),
            {"notebook": "02-04", "family": "Boosting",    "interpretable": False,
             "needs_proba": True,  "n_params": None,         "notes": "Sequential boosting"},
        ),
        (
            "Gradient Boosting",
            GradientBoostingClassifier(n_estimators=100, random_state=SEED),
            {"notebook": "02-04", "family": "Boosting",    "interpretable": False,
             "needs_proba": True,  "n_params": None,         "notes": "Residual fitting"},
        ),
        (
            "SVM (Linear)",
            SVC(kernel="linear", C=1.0, probability=True, random_state=SEED),
            {"notebook": "02-05", "family": "SVM",         "interpretable": False,
             "needs_proba": True,  "n_params": None,         "notes": "Max margin, linear"},
        ),
        (
            "SVM (RBF)",
            SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=SEED),
            {"notebook": "02-05", "family": "SVM",         "interpretable": False,
             "needs_proba": True,  "n_params": None,         "notes": "Max margin, non-linear"},
        ),
        (
            "k-NN (k=5)",
            KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
            {"notebook": "02-07", "family": "Instance",    "interpretable": True,
             "needs_proba": True,  "n_params": 0,            "notes": "Lazy learner — slow at inference"},
        ),
        (
            "Gaussian NB",
            GaussianNB(),
            {"notebook": "02-08", "family": "Probabilistic", "interpretable": True,
             "needs_proba": True,  "n_params": None,          "notes": "Very fast, assumes feature independence"},
        ),
        (
            "Soft Voting",
            VotingClassifier(
                estimators=[
                    ("dt",  DecisionTreeClassifier(max_depth=8, random_state=SEED)),
                    ("knn", KNeighborsClassifier(n_neighbors=5)),
                    ("gnb", GaussianNB()),
                    ("lr",  LogisticRegression(max_iter=500, random_state=SEED)),
                ],
                voting="soft",
            ),
            {"notebook": "02-09", "family": "Ensemble",    "interpretable": False,
             "needs_proba": True,  "n_params": None,         "notes": "4-model soft vote"},
        ),
        (
            "Stacking",
            StackingClassifier(
                estimators=[
                    ("dt",  DecisionTreeClassifier(max_depth=8, random_state=SEED)),
                    ("knn", KNeighborsClassifier(n_neighbors=5)),
                    ("gnb", GaussianNB()),
                    ("lr",  LogisticRegression(max_iter=500, random_state=SEED)),
                ],
                final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
                cv=5,
            ),
            {"notebook": "02-09", "family": "Ensemble",    "interpretable": False,
             "needs_proba": True,  "n_params": None,         "notes": "OOF meta-learner"},
        ),
    ]
    return registry


registry = build_algorithm_registry()
print(f"Registered {len(registry)} algorithms for comparison.")
for name, _, meta in registry:
    print(f"  [{meta['notebook']}] {name:<22} family={meta['family']}")

---

## Part 2 — Training All Algorithms on Digits

We fit every algorithm on the same `(X_train, y_train)` and evaluate on `(X_test, y_test)`. Training time is measured with multiple repeats for stability; inference time is measured separately on the test set.

In [ ]:
def time_fit(
    clf: object,
    X_train: np.ndarray,
    y_train: np.ndarray,
    n_repeats: int = 3,
) -> tuple:
    """Fit classifier and return (fitted_clf, mean_train_time_seconds).

    Args:
        clf: Unfitted sklearn-compatible classifier.
        X_train: Training feature matrix.
        y_train: Training labels.
        n_repeats: Number of timing repetitions (best-of is NOT used; mean is returned).

    Returns:
        Tuple of (fitted_clf, mean_train_time).
    """
    times = []
    fitted = None
    for _ in range(n_repeats):
        clf_copy = copy.deepcopy(clf)
        t0 = time.perf_counter()
        clf_copy.fit(X_train, y_train)
        times.append(time.perf_counter() - t0)
        fitted = clf_copy
    return fitted, float(np.mean(times))


def time_predict(
    clf: object,
    X_test: np.ndarray,
    n_repeats: int = 5,
) -> tuple:
    """Measure mean inference time for predict() over n_repeats.

    Args:
        clf: Fitted sklearn-compatible classifier.
        X_test: Test feature matrix.
        n_repeats: Number of timing repetitions.

    Returns:
        Tuple of (predictions, mean_inference_time_seconds).
    """
    preds = None
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        preds = clf.predict(X_test)
        times.append(time.perf_counter() - t0)
    return preds, float(np.mean(times))

In [ ]:
# ── Run the full comparison ───────────────────────────────────────────────────
print("Training all algorithms... (this may take ~60s for boosting + SVM)")
print("=" * 65)

comparison_records: list = []
fitted_clfs: dict = {}

for name, clf, meta in registry:
    fitted_clf, train_time = time_fit(clf, X_train, y_train, n_repeats=N_REPEATS)
    preds, infer_time = time_predict(fitted_clf, X_test, n_repeats=5)

    acc  = accuracy_score(y_test, preds)
    f1   = f1_score(y_test, preds, average="macro")

    # Probability-based metrics (where available)
    if meta["needs_proba"] and hasattr(fitted_clf, "predict_proba"):
        proba = fitted_clf.predict_proba(X_test)
        roc_auc = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
    else:
        roc_auc = float("nan")

    record = {
        "Algorithm":        name,
        "Notebook":         meta["notebook"],
        "Family":           meta["family"],
        "Acc":              acc,
        "F1 Macro":         f1,
        "ROC-AUC (OvR)":    roc_auc,
        "Train Time (s)":   train_time,
        "Infer Time (ms)":  infer_time * 1000,   # convert to ms
        "Interpretable":    meta["interpretable"],
        "Notes":            meta["notes"],
    }
    comparison_records.append(record)
    fitted_clfs[name] = fitted_clf

    print(
        f"  {name:<22} | Acc={acc:.4f} | F1={f1:.4f} | "
        f"Train={train_time:.2f}s | Infer={infer_time*1000:.1f}ms"
    )

results_df = (
    pd.DataFrame(comparison_records)
    .sort_values("Acc", ascending=False)
    .reset_index(drop=True)
)
print("\nDone.")

---

## Part 3 — Side-by-Side Comparison

### 3.1 Full Results Table

All 13 algorithms ranked by test accuracy.

In [ ]:
# Display full results table
display_df = results_df[[
    "Algorithm", "Family", "Acc", "F1 Macro",
    "ROC-AUC (OvR)", "Train Time (s)", "Infer Time (ms)", "Interpretable",
]].copy()
display_df["Acc"]              = display_df["Acc"].map(lambda x: f"{x:.4f}")
display_df["F1 Macro"]        = display_df["F1 Macro"].map(lambda x: f"{x:.4f}")
display_df["ROC-AUC (OvR)"]   = display_df["ROC-AUC (OvR)"].map(
    lambda x: f"{x:.4f}" if not (isinstance(x, float) and x != x) else "—"
)
display_df["Train Time (s)"]  = display_df["Train Time (s)"].map(lambda x: f"{x:.3f}")
display_df["Infer Time (ms)"] = display_df["Infer Time (ms)"].map(lambda x: f"{x:.2f}")
display_df["Interpretable"]   = display_df["Interpretable"].map(lambda x: "Yes" if x else "No")

print(display_df.to_string(index=False))

In [ ]:
# Accuracy bar chart — all algorithms
acc_sorted = results_df.sort_values("Acc", ascending=True)
family_colors = {
    "Baseline":     "#AAAAAA",
    "Linear":       "#5B9BD5",
    "Tree":         "#70AD47",
    "Ensemble":     "#2E75B6",
    "Boosting":     "#375623",
    "SVM":          "#ED7D31",
    "Instance":     "#FFC000",
    "Probabilistic":"#9E480E",
}
colors = [family_colors.get(f, "#333333") for f in acc_sorted["Family"]]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    acc_sorted["Algorithm"], acc_sorted["Acc"],
    color=colors, edgecolor="black", linewidth=0.5,
)
ax.set_xlabel("Test Accuracy")
ax.set_title("All Module 02 Algorithms — Digits Test Accuracy")
ax.set_xlim(0.05, 1.05)

for bar, acc in zip(bars, acc_sorted["Acc"]):
    ax.text(
        acc + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{acc:.3f}", va="center", fontsize=8,
    )

# Legend by family
legend_patches = [
    mpatches.Patch(color=c, label=f) for f, c in family_colors.items()
]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

### 3.2 Speed vs Accuracy Trade-off

A scatter plot of training time versus test accuracy reveals the trade-off frontier. Algorithms in the top-left are preferred: high accuracy, low training cost.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for _, row in results_df.iterrows():
    color = family_colors.get(row["Family"], "#333333")
    ax.scatter(
        row["Train Time (s)"], row["Acc"],
        s=120, color=color, edgecolor="black", linewidth=0.7, zorder=3,
    )
    ax.annotate(
        row["Algorithm"],
        xy=(row["Train Time (s)"], row["Acc"]),
        xytext=(5, 3), textcoords="offset points",
        fontsize=7.5,
    )

ax.set_xscale("log")
ax.set_xlabel("Training Time (s, log scale)")
ax.set_ylabel("Test Accuracy")
ax.set_title("Speed vs Accuracy Trade-off — Module 02 Algorithms")
ax.grid(True, linestyle="--", alpha=0.4)

legend_patches = [
    mpatches.Patch(color=c, label=f) for f, c in family_colors.items()
    if f in results_df["Family"].values
]
ax.legend(handles=legend_patches, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Inference time comparison (log scale)
inf_sorted = results_df.sort_values("Infer Time (ms)", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
inf_colors = [family_colors.get(f, "#333333") for f in inf_sorted["Family"]]
bars = ax.barh(
    inf_sorted["Algorithm"],
    inf_sorted["Infer Time (ms)"],
    color=inf_colors,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_xscale("log")
ax.set_xlabel("Inference Time (ms, log scale) — full test set")
ax.set_title("Inference Latency — All Module 02 Algorithms")
for bar, t in zip(bars, inf_sorted["Infer Time (ms)"]):
    ax.text(
        t * 1.05, bar.get_y() + bar.get_height() / 2,
        f"{t:.2f}ms", va="center", fontsize=8,
    )
plt.tight_layout()
plt.show()

### 3.3 Cross-Validation Accuracy

A single train/test split can be noisy. We verify rankings with 5-fold stratified CV on the full dataset.

In [ ]:
# 5-fold CV on full dataset for robustness
print("5-fold CV accuracy (full dataset, this may take ~60s):")
cv_results: dict = {}
for name, _, _ in registry:
    clf_fresh = copy.deepcopy(dict(
        zip([n for n, _, _ in registry],
            [c for _, c, _ in registry])
    )[name])
    scores = cross_val_score(
        clf_fresh, X_all, y_all,
        cv=StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED),
        scoring="accuracy",
        n_jobs=1,
    )
    cv_results[name] = scores
    print(f"  {name:<22} mean={scores.mean():.4f}  std={scores.std():.4f}")

# Compare single-split vs CV mean
cv_df = pd.DataFrame([
    {
        "Algorithm":      name,
        "Test Acc (single)": results_df.set_index("Algorithm").loc[name, "Acc"],
        "CV Mean":  cv_results[name].mean(),
        "CV Std":   cv_results[name].std(),
    }
    for name in [n for n, _, _ in registry]
]).sort_values("CV Mean", ascending=False).reset_index(drop=True)

print("\nRanking stability (single split vs 5-fold CV):")
print(cv_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

---

## Part 4 — Tradeoff Analysis & Decision Framework

### 4.1 Multi-Dimensional Radar Chart

No single metric captures algorithm quality across all deployment scenarios. We visualise five dimensions simultaneously: accuracy, speed (inverted training time), interpretability, probability calibration quality, and inference latency (inverted).

In [ ]:
def normalise_0_to_1(values: list) -> list:
    """Min-max normalise a list of floats to [0, 1].

    Args:
        values: Input float values.

    Returns:
        Normalised values in [0, 1].
    """
    arr = np.array(values, dtype=float)
    rng = arr.max() - arr.min()
    if rng == 0:
        return [0.5] * len(values)
    return list((arr - arr.min()) / rng)


# Select a representative subset for the radar (avoid clutter)
radar_names = [
    "Majority Baseline", "Logistic Regression", "Decision Tree",
    "Random Forest", "Gradient Boosting", "SVM (RBF)",
    "k-NN (k=5)", "Stacking",
]
radar_df = results_df[results_df["Algorithm"].isin(radar_names)].copy()
radar_df = radar_df.set_index("Algorithm").loc[radar_names].reset_index()

# Compute radar dimensions
acc_norm     = normalise_0_to_1(radar_df["Acc"].tolist())
speed_norm   = normalise_0_to_1(
    [1.0 / (t + 1e-6) for t in radar_df["Train Time (s)"].tolist()]
)
interp_norm  = [1.0 if v else 0.0 for v in
                [results_df.set_index("Algorithm").loc[n, "Interpretable"]
                 for n in radar_names]]
latency_norm = normalise_0_to_1(
    [1.0 / (t + 1e-6) for t in radar_df["Infer Time (ms)"].tolist()]
)
f1_norm      = normalise_0_to_1(radar_df["F1 Macro"].tolist())

categories   = ["Accuracy", "Train Speed", "Interpretable", "Infer Speed", "F1"]
n_cats       = len(categories)
angles       = [n / float(n_cats) * 2 * np.pi for n in range(n_cats)]
angles      += angles[:1]   # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
radar_palette = plt.cm.tab10(np.linspace(0, 1, len(radar_names)))

for idx, name in enumerate(radar_names):
    values = [
        acc_norm[idx], speed_norm[idx], interp_norm[idx],
        latency_norm[idx], f1_norm[idx],
    ]
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, label=name, color=radar_palette[idx])
    ax.fill(angles, values, alpha=0.06, color=radar_palette[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_yticks([0.25, 0.5, 0.75])
ax.set_yticklabels(["0.25", "0.50", "0.75"], size=8)
ax.set_title("Multi-Dimensional Algorithm Comparison (normalised)", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)
plt.tight_layout()
plt.show()

### 4.2 Error Analysis — Where Do Algorithms Fail?

Each algorithm fails on a different *subset* of test samples. We compute per-sample error frequency across all 13 models to identify the *hardest* samples — those that no algorithm gets right.

In [ ]:
def per_sample_error_analysis(
    fitted_clfs: dict,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> tuple:
    """Count how many algorithms fail on each test sample.

    Args:
        fitted_clfs: Dict mapping algorithm name to fitted estimator.
        X_test: Test feature matrix.
        y_test: True test labels.

    Returns:
        Tuple of (error_counts array shape (n_test,),
                  pct_of_models_failing per sample).
    """
    n_test = len(y_test)
    error_counts = np.zeros(n_test, dtype=int)
    for name, clf in fitted_clfs.items():
        preds = clf.predict(X_test)
        errors = (preds != y_test)
        error_counts += errors.astype(int)
    return error_counts, error_counts / len(fitted_clfs)


error_counts, error_pct = per_sample_error_analysis(fitted_clfs, X_test, y_test)

# Histogram of error counts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(error_counts, bins=range(0, len(fitted_clfs) + 2),
             color="#4472C4", edgecolor="black", rwidth=0.8)
axes[0].set_xlabel("Number of algorithms wrong on sample")
axes[0].set_ylabel("Number of test samples")
axes[0].set_title("Per-Sample Error Count Distribution")

# Identify the hardest samples (failed by > 50% of models)
hard_idx = np.where(error_pct > 0.5)[0]
n_hard   = len(hard_idx)
axes[1].bar(
    ["Easy (0 errors)", "Partial (1–6 errors)", "Hard (>50% models wrong)"],
    [
        (error_counts == 0).sum(),
        ((error_counts > 0) & (error_counts <= len(fitted_clfs) // 2)).sum(),
        n_hard,
    ],
    color=["#70AD47", "#FFC000", "#C00000"],
    edgecolor="black",
)
axes[1].set_ylabel("Number of test samples")
axes[1].set_title("Sample Difficulty Tiers")

plt.tight_layout()
plt.show()

# Visualise the hardest digits
if n_hard > 0:
    show_k = min(8, n_hard)
    hardest_sorted = hard_idx[np.argsort(error_pct[hard_idx])[::-1]][:show_k]
    fig2, axes2 = plt.subplots(1, show_k, figsize=(2 * show_k, 2.5))
    if show_k == 1:
        axes2 = [axes2]
    for ax, si in zip(axes2, hardest_sorted):
        img = X_test_raw[si].reshape(8, 8)
        ax.imshow(img, cmap="gray_r")
        ax.set_title(
            f"True:{y_test[si]}\n{error_counts[si]}/{len(fitted_clfs)} wrong",
            fontsize=8,
        )
        ax.axis("off")
    plt.suptitle("Hardest Test Samples (most models fail)", fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f"\n{n_hard} samples failed by >50% of models.")
else:
    print("No sample failed by more than 50% of models — strong overall agreement.")

### 4.3 Per-Class Performance Breakdown

Overall accuracy obscures per-class strengths. Some digits are universally easy (0, 1); others are systematically harder (8, 9). We compute per-class F1 for the top 5 models.

In [ ]:
def per_class_f1(
    fitted_clf: object,
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int = 10,
) -> np.ndarray:
    """Compute per-class F1 score for a fitted classifier.

    Args:
        fitted_clf: Fitted sklearn-compatible classifier.
        X_test: Test feature matrix.
        y_test: True test labels.
        n_classes: Number of classes.

    Returns:
        Array of per-class F1 scores, shape (n_classes,).
    """
    preds = fitted_clf.predict(X_test)
    report = classification_report(y_test, preds, output_dict=True)
    return np.array([report[str(c)]["f1-score"] for c in range(n_classes)])


# Select top 5 models by accuracy
top5_names = results_df.head(5)["Algorithm"].tolist()
class_f1_matrix = np.zeros((len(top5_names), 10))
for row_idx, name in enumerate(top5_names):
    class_f1_matrix[row_idx] = per_class_f1(fitted_clfs[name], X_test, y_test)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(10)
bar_width = 0.15
offset_start = -(len(top5_names) - 1) * bar_width / 2
cmap_top5 = plt.cm.Set2(np.linspace(0, 1, len(top5_names)))

for i, (name, c) in enumerate(zip(top5_names, cmap_top5)):
    ax.bar(
        x + offset_start + i * bar_width,
        class_f1_matrix[i],
        width=bar_width,
        label=name,
        color=c,
        edgecolor="black",
        linewidth=0.4,
    )

ax.set_xticks(x)
ax.set_xticklabels([f"Digit {d}" for d in range(10)], rotation=30)
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 Score — Top 5 Algorithms")
ax.set_ylim(0.7, 1.02)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 4.4 Algorithm Selection Decision Framework

The right algorithm depends on four key axes:

| Axis | Key question |
|------|-------------|
| **Data size** | How many training samples do I have? |
| **Dimensionality** | How many features vs samples? |
| **Interpretability** | Does the decision need to be explainable? |
| **Latency** | How fast must training and inference be? |

Use the decision guide below as a starting point — always validate with cross-validation on your specific data.

In [ ]:
def algorithm_selection_guide(
    n_samples: int,
    n_features: int,
    need_interpretable: bool,
    max_train_seconds: float,
    max_infer_ms_per_batch: float,
    need_probabilities: bool,
) -> list:
    """Recommend algorithms based on dataset and deployment constraints.

    This heuristic guide distils the Module 02 comparison results into
    practical selection rules. Always validate recommendations empirically.

    Args:
        n_samples: Number of training samples.
        n_features: Number of input features.
        need_interpretable: True if human-readable decisions are required.
        max_train_seconds: Maximum acceptable training wall-clock time.
        max_infer_ms_per_batch: Maximum inference latency (ms) per batch.
        need_probabilities: True if calibrated class probabilities are needed.

    Returns:
        List of recommended algorithm names in priority order.
    """
    candidates: list = []

    # Interpretability filter
    if need_interpretable:
        candidates = [
            "Logistic Regression",  # linear, coefficients inspectable
            "Decision Tree",         # tree structure readable
            "Ridge Classifier",      # linear weights
            "Gaussian NB",           # per-class likelihoods
            "k-NN (k=5)",            # retrieve nearest neighbours
        ]
    else:
        candidates = [
            "Stacking", "Random Forest", "Gradient Boosting",
            "SVM (RBF)", "SVM (Linear)", "Soft Voting",
            "Logistic Regression", "k-NN (k=5)", "Gaussian NB",
            "Decision Tree",
        ]

    # Small dataset heuristics (< 500 samples) — avoid high-variance models
    if n_samples < 500:
        remove = {"Random Forest", "Gradient Boosting", "AdaBoost", "Stacking"}
        candidates = [c for c in candidates if c not in remove]
        if "k-NN (k=5)" not in candidates:
            candidates.insert(0, "k-NN (k=5)")

    # High dimensionality (features >> samples) — linear models preferred
    if n_features > n_samples:
        prefer_linear = ["Ridge Classifier", "Logistic Regression", "SVM (Linear)"]
        for alg in prefer_linear:
            if alg not in candidates:
                candidates.insert(0, alg)

    # Training time filter
    mean_train_times = results_df.set_index("Algorithm")["Train Time (s)"]
    candidates = [
        c for c in candidates
        if c in mean_train_times.index
        and mean_train_times[c] <= max_train_seconds * 10  # scale to dataset
    ]

    # Inference latency filter
    mean_infer_times = results_df.set_index("Algorithm")["Infer Time (ms)"]
    candidates = [
        c for c in candidates
        if c in mean_infer_times.index
        and mean_infer_times[c] <= max_infer_ms_per_batch * 5
    ]

    # Probability requirement filter
    if need_probabilities:
        no_prob = {"Ridge Classifier"}
        candidates = [c for c in candidates if c not in no_prob]

    return candidates[:5] if candidates else ["Logistic Regression"]


# Demo: run the guide for several deployment scenarios
scenarios = [
    {
        "label": "Small medical dataset, need explainability",
        "kwargs": {
            "n_samples": 300, "n_features": 20,
            "need_interpretable": True, "max_train_seconds": 60.0,
            "max_infer_ms_per_batch": 100.0, "need_probabilities": True,
        },
    },
    {
        "label": "Large production system, speed matters",
        "kwargs": {
            "n_samples": 50_000, "n_features": 100,
            "need_interpretable": False, "max_train_seconds": 300.0,
            "max_infer_ms_per_batch": 5.0, "need_probabilities": False,
        },
    },
    {
        "label": "High-dim text features (sparse)",
        "kwargs": {
            "n_samples": 5_000, "n_features": 50_000,
            "need_interpretable": False, "max_train_seconds": 120.0,
            "max_infer_ms_per_batch": 20.0, "need_probabilities": True,
        },
    },
    {
        "label": "Real-time edge device, minimal latency",
        "kwargs": {
            "n_samples": 1_000, "n_features": 30,
            "need_interpretable": True, "max_train_seconds": 10.0,
            "max_infer_ms_per_batch": 1.0, "need_probabilities": False,
        },
    },
]

print("Algorithm Selection Guide — Scenario Recommendations")
print("=" * 60)
for scenario in scenarios:
    recs = algorithm_selection_guide(**scenario["kwargs"])
    print(f"\n  Scenario: {scenario['label']}")
    for rank, alg in enumerate(recs, 1):
        print(f"    {rank}. {alg}")

### 4.5 Confusion Matrix Grid — Top 4 Algorithms

Side-by-side confusion matrices reveal where each algorithm diverges in its error patterns.

In [ ]:
top4_names = results_df.head(4)["Algorithm"].tolist()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, top4_names):
    clf = fitted_clfs[name]
    preds = clf.predict(X_test)
    cm_mat = confusion_matrix(y_test, preds)
    im = ax.imshow(cm_mat, cmap="Blues")
    ax.set_title(f"{name}\nAcc={accuracy_score(y_test, preds):.3f}", fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    thresh = cm_mat.max() / 2
    for i in range(10):
        for j in range(10):
            ax.text(j, i, str(cm_mat[i, j]),
                    ha="center", va="center", fontsize=6,
                    color="white" if cm_mat[i, j] > thresh else "black")

plt.suptitle("Confusion Matrices — Top 4 Algorithms", fontsize=11)
plt.tight_layout()
plt.show()

### 4.6 Learning Curves — Accuracy vs Training Set Size

Learning curves show how each algorithm responds to more training data. Models with high bias (Naive Bayes, Ridge) plateau early; high-variance models (Decision Tree) improve more with data.

In [ ]:
def learning_curve_experiment(
    clf_factory: object,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    fractions: list,
    seed: int = SEED,
) -> list:
    """Measure test accuracy as a function of training set fraction.

    Args:
        clf_factory: Unfitted classifier (will be deep-copied for each fraction).
        X_train: Full training feature matrix.
        y_train: Full training labels.
        X_test: Test feature matrix.
        y_test: True test labels.
        fractions: List of fractions in (0, 1] to use from the training set.
        seed: Random state for subsampling.

    Returns:
        List of test accuracies corresponding to each fraction.
    """
    rng = np.random.RandomState(seed)
    accs = []
    for frac in fractions:
        n = max(int(frac * len(X_train)), len(np.unique(y_train)))
        idx = rng.choice(len(X_train), size=n, replace=False)
        clf_copy = copy.deepcopy(clf_factory)
        clf_copy.fit(X_train[idx], y_train[idx])
        accs.append(accuracy_score(y_test, clf_copy.predict(X_test)))
    return accs


fractions = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
lc_algorithms = {
    "Logistic Regression":  LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    "Decision Tree":         DecisionTreeClassifier(max_depth=10, random_state=SEED),
    "Random Forest":         RandomForestClassifier(n_estimators=50, random_state=SEED),
    "Gaussian NB":           GaussianNB(),
    "k-NN (k=5)":           KNeighborsClassifier(n_neighbors=5),
}

fig, ax = plt.subplots(figsize=(9, 5))
lc_palette = plt.cm.tab10(np.linspace(0, 1, len(lc_algorithms)))

n_samples_axis = [int(f * X_train.shape[0]) for f in fractions]
for (name, clf), c in zip(lc_algorithms.items(), lc_palette):
    accs = learning_curve_experiment(clf, X_train, y_train, X_test, y_test, fractions)
    ax.plot(n_samples_axis, accs, marker="o", label=name, color=c, linewidth=2)

ax.set_xlabel("Training samples used")
ax.set_ylabel("Test Accuracy")
ax.set_title("Learning Curves — Accuracy vs Training Set Size")
ax.legend(fontsize=8)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

### 4.7 Ranking Stability — Robustness to Data Subsets

We test whether the accuracy ranking is stable across 5 random 80/20 splits. An algorithm that ranks #1 on only one split but drops to #5 on others should not be chosen purely on single-split evidence.

In [ ]:
def ranking_stability_experiment(
    registry: list,
    X_all: np.ndarray,
    y_all: np.ndarray,
    n_splits: int = 5,
    test_size: float = 0.2,
) -> pd.DataFrame:
    """Compute accuracy ranking across multiple random train/test splits.

    Args:
        registry: List of (name, estimator, metadata) tuples.
        X_all: Full feature matrix.
        y_all: Full label vector.
        n_splits: Number of random splits to evaluate.
        test_size: Fraction of data for test in each split.

    Returns:
        DataFrame with algorithms as rows, splits as columns, values are ranks.
    """
    all_accs: dict = {name: [] for name, _, _ in registry}

    for split_idx in range(n_splits):
        seed_s = SEED + split_idx * 17
        scaler_s = StandardScaler()
        Xtr, Xte, ytr, yte = train_test_split(
            X_all, y_all, test_size=test_size,
            random_state=seed_s, stratify=y_all,
        )
        Xtr = scaler_s.fit_transform(Xtr)
        Xte = scaler_s.transform(Xte)

        for name, clf, _ in registry:
            clf_copy = copy.deepcopy(clf)
            clf_copy.fit(Xtr, ytr)
            acc = accuracy_score(yte, clf_copy.predict(Xte))
            all_accs[name].append(acc)

        print(f"  Split {split_idx + 1}/{n_splits} done")

    # Convert to rank per split
    split_accs_df = pd.DataFrame(all_accs)
    rank_df = split_accs_df.rank(ascending=False, axis=1).astype(int)
    mean_rank = rank_df.mean(axis=0)
    mean_acc  = split_accs_df.mean(axis=0)
    std_acc   = split_accs_df.std(axis=0)

    summary = pd.DataFrame({
        "Algorithm":   list(all_accs.keys()),
        "Mean Acc":    [mean_acc[n] for n in all_accs],
        "Std Acc":     [std_acc[n] for n in all_accs],
        "Mean Rank":   [mean_rank[n] for n in all_accs],
    }).sort_values("Mean Rank").reset_index(drop=True)
    return summary


print("Ranking stability across 5 random splits:")
stability_df = ranking_stability_experiment(registry, X_all, y_all)
print(stability_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Visualise rank stability
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    stability_df["Algorithm"][::-1],
    stability_df["Mean Acc"][::-1],
    xerr=stability_df["Std Acc"][::-1],
    color="#4472C4", edgecolor="black", linewidth=0.5,
    capsize=4,
)
ax.set_xlabel("Mean Accuracy ± Std (5 random splits)")
ax.set_title("Ranking Stability — Mean Accuracy Across 5 Random 80/20 Splits")
plt.tight_layout()
plt.show()

### 4.8 Feature Importance Across Tree-Based Models

Tree-based models (Decision Tree, Random Forest, Gradient Boosting) expose per-feature importance scores via impurity decrease. We compare which pixel regions each model considers most informative for digit classification.

In [ ]:
def extract_feature_importance(
    clf: object,
    n_features: int,
) -> np.ndarray | None:
    """Extract feature importances from a tree-based classifier.

    Returns None for models that do not expose importances.

    Args:
        clf: Fitted sklearn-compatible classifier.
        n_features: Total number of input features.

    Returns:
        Importance array of shape (n_features,), or None if unavailable.
    """
    if hasattr(clf, "feature_importances_"):
        return clf.feature_importances_
    if hasattr(clf, "coef_"):
        # For linear models, use L2 norm of coefficient vector per feature
        coef = np.abs(clf.coef_)
        if coef.ndim == 2:
            return coef.mean(axis=0)
        return coef
    return None


importance_models = {
    "Decision Tree":     fitted_clfs["Decision Tree"],
    "Random Forest":     fitted_clfs["Random Forest"],
    "Gradient Boosting": fitted_clfs["Gradient Boosting"],
    "Logistic Regression": fitted_clfs["Logistic Regression"],
}

fig, axes = plt.subplots(1, len(importance_models), figsize=(14, 3))
for ax, (name, clf) in zip(axes, importance_models.items()):
    imp = extract_feature_importance(clf, X_train.shape[1])
    if imp is None:
        ax.text(0.5, 0.5, "Not available", ha="center", va="center")
        ax.set_title(name)
        continue
    imp_img = imp.reshape(8, 8)
    # Normalise to [0, 1] for visual consistency
    imp_img = (imp_img - imp_img.min()) / (imp_img.max() - imp_img.min() + 1e-12)
    im = ax.imshow(imp_img, cmap="hot")
    ax.set_title(f"{name}\n(importance heatmap)")
    ax.axis("off")

plt.suptitle("Feature Importance Heatmaps (8x8 pixel grid)", fontsize=11)
plt.tight_layout()
plt.show()
print("Brighter = more important pixel region for classification.")
print("Tree methods focus on central ink pixels; linear models use edge structure.")

### 4.9 Probability Calibration — Reliability Diagrams

A well-calibrated model's predicted probabilities match empirical frequencies: when the model says 80% confidence, it should be correct ~80% of the time. We compare reliability diagrams for models that expose `predict_proba`.

In [ ]:
def reliability_diagram(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    n_bins: int = 10,
) -> tuple:
    """Compute reliability diagram data for a multi-class classifier.

    Uses a one-vs-rest approach: for each sample, the confidence is the
    maximum class probability and the outcome is whether that class is correct.

    Args:
        y_true: True labels, shape (n_samples,).
        y_proba: Predicted probabilities, shape (n_samples, n_classes).
        n_bins: Number of confidence bins.

    Returns:
        Tuple of (bin_centres, bin_acc, bin_counts) arrays.
    """
    max_proba = y_proba.max(axis=1)
    predicted = y_proba.argmax(axis=1)
    correct = (predicted == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centres = (bins[:-1] + bins[1:]) / 2
    bin_acc    = np.zeros(n_bins)
    bin_counts = np.zeros(n_bins, dtype=int)

    for b in range(n_bins):
        mask = (max_proba >= bins[b]) & (max_proba < bins[b + 1])
        if mask.sum() > 0:
            bin_acc[b]    = correct[mask].mean()
            bin_counts[b] = mask.sum()

    return bin_centres, bin_acc, bin_counts


def compute_expected_calibration_error(
    bin_centres: np.ndarray,
    bin_acc: np.ndarray,
    bin_counts: np.ndarray,
) -> float:
    """Compute Expected Calibration Error (ECE) from reliability diagram data.

    ECE = sum over bins of (|bin_count / n| * |bin_accuracy - bin_confidence|).

    Args:
        bin_centres: Confidence midpoints per bin.
        bin_acc: Mean accuracy per confidence bin.
        bin_counts: Sample count per bin.

    Returns:
        Expected Calibration Error in [0, 1] (lower is better).
    """
    n_total = bin_counts.sum()
    if n_total == 0:
        return float("nan")
    ece = np.sum(
        (bin_counts / n_total) * np.abs(bin_acc - bin_centres)
    )
    return float(ece)


proba_models = {
    name: clf
    for name, clf in fitted_clfs.items()
    if hasattr(clf, "predict_proba")
    and name not in {"Ridge Classifier", "Majority Baseline"}
}

# Select up to 4 for display
display_proba_models = dict(list(proba_models.items())[:4])

fig, axes = plt.subplots(1, len(display_proba_models), figsize=(14, 4))
if len(display_proba_models) == 1:
    axes = [axes]

ece_scores: dict = {}
for ax, (name, clf) in zip(axes, display_proba_models.items()):
    proba = clf.predict_proba(X_test)
    centres, acc, counts = reliability_diagram(y_test, proba, n_bins=10)
    ece = compute_expected_calibration_error(centres, acc, counts)
    ece_scores[name] = ece

    ax.bar(centres, acc, width=0.08, alpha=0.7, color="#4472C4",
           edgecolor="black", label="Classifier")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{name}\nECE={ece:.3f}")
    ax.legend(fontsize=7)

plt.suptitle("Reliability Diagrams — Probability Calibration", fontsize=11)
plt.tight_layout()
plt.show()

print("Expected Calibration Error (ECE) — lower is better:")
for name, ece in sorted(ece_scores.items(), key=lambda kv: kv[1]):
    print(f"  {name:<25}: ECE = {ece:.4f}")
print("\nNote: Gaussian NB is known to be poorly calibrated despite high accuracy.")

### 4.10 Model Complexity Summary

A quick reference table summarising algorithm complexity — Big-O training cost, inference cost, and memory footprint — on the Digits dataset.

In [ ]:
def build_complexity_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    """Build a human-readable complexity reference table.

    Combines measured train/inference times with theoretical complexity
    descriptions for all algorithms in the comparison.

    Args:
        results_df: DataFrame produced by the main comparison loop,
                    must contain Algorithm, Train Time (s), Infer Time (ms).

    Returns:
        Complexity summary DataFrame sorted by train time.
    """
    complexity_info = {
        "Majority Baseline":     {"Train O": "O(n)",        "Infer O": "O(1)",        "Memory": "O(K)"},
        "Ridge Classifier":      {"Train O": "O(n d^2)",    "Infer O": "O(d K)",      "Memory": "O(d K)"},
        "Logistic Regression":   {"Train O": "O(n d K)",    "Infer O": "O(d K)",      "Memory": "O(d K)"},
        "Decision Tree":         {"Train O": "O(n d log n)", "Infer O": "O(depth)",   "Memory": "O(nodes)"},
        "Random Forest":         {"Train O": "O(T n d log n)","Infer O": "O(T depth)","Memory": "O(T nodes)"},
        "AdaBoost":              {"Train O": "O(T n d)",    "Infer O": "O(T depth)",  "Memory": "O(T nodes)"},
        "Gradient Boosting":     {"Train O": "O(T n d)",    "Infer O": "O(T depth)",  "Memory": "O(T nodes)"},
        "SVM (Linear)":          {"Train O": "O(n^2 d)",    "Infer O": "O(sv d)",     "Memory": "O(sv d)"},
        "SVM (RBF)":             {"Train O": "O(n^3)",      "Infer O": "O(sv d)",     "Memory": "O(sv d)"},
        "k-NN (k=5)":           {"Train O": "O(1)",         "Infer O": "O(n d)",      "Memory": "O(n d)"},
        "Gaussian NB":           {"Train O": "O(n d K)",    "Infer O": "O(d K)",      "Memory": "O(d K)"},
        "Soft Voting":           {"Train O": "O(M * base)",  "Infer O": "O(M * base)","Memory": "O(M * base)"},
        "Stacking":              {"Train O": "O(F * M * base)","Infer O": "O(M + meta)","Memory": "O(M + meta)"},
    }

    rows = []
    for _, row in results_df.iterrows():
        name = row["Algorithm"]
        info = complexity_info.get(name, {"Train O": "?", "Infer O": "?", "Memory": "?"})
        rows.append({
            "Algorithm":      name,
            "Family":         row["Family"],
            "Train Complexity": info["Train O"],
            "Infer Complexity": info["Infer O"],
            "Memory":           info["Memory"],
            "Measured Train (s)": f"{row['Train Time (s)']:.3f}",
            "Measured Infer (ms)": f"{row['Infer Time (ms)']:.2f}",
        })

    return pd.DataFrame(rows)


complexity_df = build_complexity_summary(results_df)
print("Algorithm Complexity Reference (n=samples, d=features, K=classes, T=trees, F=folds, M=models, sv=support vectors)")
print()
print(complexity_df.to_string(index=False))

---

## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **No single best algorithm:** The top performers on Digits (SVM-RBF, Stacking, Random Forest, Gradient Boosting) are not universally superior — on a small dataset, a Logistic Regression or k-NN often wins after proper tuning. Always test multiple approaches.
- **Speed vs quality trade-off is real:** SVM and boosting methods achieve the highest accuracy but train 10–100× slower than Logistic Regression or Naive Bayes. For real-time or resource-constrained systems, simpler linear models are often the correct choice.
- **Interpretability costs accuracy:** The most interpretable models (Decision Tree, Logistic Regression, Naive Bayes) are uniformly outperformed by ensemble methods and SVMs on this dataset — a reflection of the dataset's non-linear structure.
- **Ensemble methods are robust but expensive:** Stacking and Soft Voting are consistently near the top across splits, but their training cost is $M 	imes N_{	ext{folds}}$ times a single model. Reserve them for cases where the accuracy gain justifies the overhead.
- **Learning curves reveal algorithm character:** High-bias algorithms (Naive Bayes, Ridge) plateau early; high-variance algorithms (Decision Tree, Gradient Boosting) benefit more from additional training data. Use learning curves to decide whether to collect more data.

### What's Next

→ **Module 03 (Unsupervised & Statistical Learning)** shifts from labelled supervision to clustering, dimensionality reduction, and density estimation — discovering structure in data without target labels.

### Going Further

- Hyperparameter tuning with `GridSearchCV` or `RandomizedSearchCV` would likely reshuffle these rankings significantly — default hyperparameters were used for fair comparison here.
- `sklearn.pipeline.Pipeline` + `sklearn.preprocessing` enables end-to-end pipelines with cross-validated hyperparameter search.
- For tabular data at scale, XGBoost and LightGBM consistently outperform vanilla Gradient Boosting with 10–50× speedup.